# Class 1.5: APIs and the web

Here you run and tweak the code.

**What we will cover**

- Reading a response body (JSON is a dict)
- Making a request with `requests`, and checking the status
- A local LLM call with Ollama
- A hosted LLM call with Groq, with the key in an environment variable

The Ollama and Groq cells call real services, so run them on your own machine (Ollama running locally, and a `GROQ_API_KEY` in your environment). The first cell runs anywhere.

## A response is a dict

When you call `.json()` on a response, you get a plain Python dict. Read it the way you read any dict.

In [6]:
# This is what response.json() gives you.
response = {"city": "Paris", "temp_c": 18, "conditions": "cloudy"}

print(f"{response['city']}: {response['temp_c']}C, {response['conditions']}")
print("humidity:", response.get("humidity", "n/a"))   # .get with a default

Paris: 18C, cloudy
humidity: n/a


## Making a request

`requests.get(url)` fetches from an endpoint; `.json()` parses the body. Always check `.status_code` before trusting the body. (Needs network; adapt the URL to any public JSON API.)

Do `pip install requests` in your virtual environment to install `requests` library.

In [7]:
import requests

resp = requests.get("https://api.github.com/users/octocat")
if resp.status_code == 200:
    data = resp.json()
    print(data)
else:
    print("request failed:", resp.status_code)

{'login': 'octocat', 'id': 583231, 'node_id': 'MDQ6VXNlcjU4MzIzMQ==', 'avatar_url': 'https://avatars.githubusercontent.com/u/583231?v=4', 'gravatar_id': '', 'url': 'https://api.github.com/users/octocat', 'html_url': 'https://github.com/octocat', 'followers_url': 'https://api.github.com/users/octocat/followers', 'following_url': 'https://api.github.com/users/octocat/following{/other_user}', 'gists_url': 'https://api.github.com/users/octocat/gists{/gist_id}', 'starred_url': 'https://api.github.com/users/octocat/starred{/owner}{/repo}', 'subscriptions_url': 'https://api.github.com/users/octocat/subscriptions', 'organizations_url': 'https://api.github.com/users/octocat/orgs', 'repos_url': 'https://api.github.com/users/octocat/repos', 'events_url': 'https://api.github.com/users/octocat/events{/privacy}', 'received_events_url': 'https://api.github.com/users/octocat/received_events', 'type': 'User', 'user_view_type': 'public', 'site_admin': False, 'name': 'The Octocat', 'company': '@github', 

## Set up Ollama (run a model on your own machine)

Ollama runs open LLMs locally, free and offline, with no API key. Install it once, pull a model, and the local cell below works. Do this in a terminal, not in the notebook.

**1. Install Ollama**

- Windows: download the installer from https://ollama.com/download/windows and run it (or in PowerShell: `irm https://ollama.com/install.ps1 | iex`).
- macOS: download the app from https://ollama.com/download and drag it into Applications (or `brew install --cask ollama`).
- Linux: `curl -fsSL https://ollama.com/install.sh | sh`

Once installed, Ollama runs in the background and serves an API at `http://localhost:11434`.

**2. Pull the model**

```
ollama pull llama3.2
```

`llama3.2` is the 3B model (about 2 GB), a good fit for a laptop. Options: `llama3.2:1b` (about 1.3 GB) if you are short on memory, or `llama3.1:8b` (about 4.7 GB) if you have the RAM.

**3. Check it works**

```
ollama run llama3.2 "Say hello in one short sentence."
```

If you get a reply, the cell below will work too. Keep Ollama running while you use the notebook.

## Local: your first LLM call with Ollama

Run this on your machine with Ollama running and a small model pulled (`ollama pull llama3.2`). Same request-and-response shape as above.

In [8]:
import requests

resp = requests.post(
    "http://localhost:11434/api/generate",
    json={"model": "llama3.2", "prompt": "Explain what an API is in two sentences.", "stream": False},
)
print(resp.json()["response"])

An Application Programming Interface (API) is a set of defined rules and protocols that enables different software systems to communicate with each other, allowing them to exchange data, services, or functionality. In essence, APIs act as messengers between systems, enabling the creation of new applications, services, and integrations by providing a standardized way for different systems to interact with each other.


## Streaming the reply
The call above used `stream: False`, so Ollama returns the whole reply as one JSON object and `resp.json()["response"]` reads it in a single step.
With `stream: True`, Ollama sends the reply piece by piece, one JSON object per chunk, ending with `"done": true`. That is how a live "typing" effect works, and it feels faster for long answers. The body is no longer a single JSON object, so `resp.json()` would fail; you read it line by line and print each chunk as it arrives.

In [11]:
import requests, json

# stream=True: read the reply piece by piece as the model writes it
with requests.post(
    "http://localhost:11434/api/generate",
    json={"model": "llama3.2", "prompt": "Explain what an API is in details.", "stream": True},
    stream=True,                       # also tell requests not to buffer the whole body
) as resp:
    for line in resp.iter_lines():
        if line:
            chunk = json.loads(line)                    # each line is its own JSON object
            print(chunk.get("response", ""), end="", flush=True)   # print as it arrives
            if chunk.get("done"):
                break
print()   # newline after the streamed text


An Application Programming Interface (API) is a set of defined rules that enables different applications, systems, or services to communicate with each other efficiently and effectively. APIs allow data, functionality, or services to be shared between systems, while maintaining the integrity and consistency of the data.

Here's a detailed explanation of what an API is:

**Components of an API**

1. **Request**: A request is a message sent by an application or system to another application or system, asking for specific data, functionality, or service.
2. **Response**: The response is the data, information, or result received from the receiving system in response to the request.
3. **Interface**: An API defines the interface through which requests are sent and responses are returned. This includes the format of the data, the operations supported, and any authentication or authorization requirements.

**Types of APIs**

1. **Web API**: A web API is a type of API that communicates over HT

## Hosted: the same prompt on Groq

Needs a free `GROQ_API_KEY` in your environment (never in the code, never committed). Groq is faster and runs bigger open models than a laptop.

In [12]:
# Get the API key from the user without echoing it to the console
import getpass
import os

if not os.environ.get('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = getpass.getpass("Enter your Groq API key: ")

In [13]:
import requests

resp = requests.post(
    "https://api.groq.com/openai/v1/chat/completions",
    headers={"Authorization": f"Bearer {os.environ['GROQ_API_KEY']}"},
    json={
        "model": "llama-3.1-8b-instant",   # check Groq's docs for a current model id
        "messages": [{"role": "user", "content": "Explain what an API is in two sentences."}],
    },
)
print(resp.json()["choices"][0]["message"]["content"])

An API, or Application Programming Interface, is a set of defined rules and protocols that enables different software systems to communicate with each other by exchanging data. It acts as a messenger between two systems, allowing them to retrieve, update, or manipulate data in a standardized and secure way, without exposing the underlying code or technical details.


## Your turn

**Micro-assignment.** Parse a response and check a status (run anywhere), then make a local and a hosted call (run locally). See `../micro-assignment/README.md`.

**Module 1 milestone assignment.** A tool that reads a dataset, computes facts with Pandas, asks an LLM to summarize them, and saves the facts plus the summary to JSON. See the `milestone-assignment/` folder.

**Before class 2.2:** watch the curated math-intuition primer.